# Data Exploration & Preprocessing

Every ML project starts with understanding your data. This notebook covers the complete data preprocessing pipeline:

1. **Exploratory Data Analysis (EDA)** - Understand distributions, relationships, and anomalies
2. **Handling Missing Values** - Strategies beyond simple imputation
3. **Encoding Categorical Variables** - Label, one-hot, ordinal, and target encoding
4. **Feature Scaling** - StandardScaler, MinMaxScaler, RobustScaler
5. **Train/Test Splitting** - Proper data partitioning to avoid leakage

**Dataset**: Titanic (binary classification) - predict passenger survival

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Load and Inspect the Data

In [ ]:
# Load Titanic dataset from seaborn's built-in datasets
df = sns.load_dataset("titanic")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary for numerical features
df.describe()

In [ ]:
# Statistical summary for categorical features
df.describe(include="object")

## 2. Missing Values Analysis

Understanding the pattern of missingness is critical - data can be:
- **MCAR** (Missing Completely at Random): missingness is unrelated to any variable
- **MAR** (Missing at Random): missingness depends on observed variables
- **MNAR** (Missing Not at Random): missingness depends on the missing value itself

In [ ]:
# Percentage of missing values per column
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
print("Missing values (%):\n")
print(missing_pct.round(1))

In [ ]:
# Visualize missing value patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of missing percentages
missing_pct.plot(kind="bar", ax=axes[0], color="coral")
axes[0].set_title("Missing Values by Feature")
axes[0].set_ylabel("% Missing")

# Heatmap of missingness pattern
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, ax=axes[1], cmap="viridis")
axes[1].set_title("Missing Value Patterns")

plt.tight_layout()
plt.show()

### Handling Missing Values

| Strategy | When to Use |
|----------|------------|
| Drop rows | Very few missing values (<5%), MCAR |
| Drop column | >50% missing, low predictive value |
| Mean/Median imputation | Numerical, MCAR/MAR |
| Mode imputation | Categorical features |
| KNN imputation | When feature correlations exist |
| Indicator variable | When missingness itself is informative |

In [ ]:
# Work with a subset of useful features
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "class", "alone"]
target = "survived"

df_clean = df[features + [target]].copy()

# Strategy 1: Median imputation for age (robust to outliers)
median_imputer = SimpleImputer(strategy="median")
df_clean["age"] = median_imputer.fit_transform(df_clean[["age"]])

# Strategy 2: Mode imputation for embarked
mode_imputer = SimpleImputer(strategy="most_frequent")
df_clean["embarked"] = mode_imputer.fit_transform(df_clean[["embarked"]]).ravel()

# Strategy 3: Add a missingness indicator before imputing (when missingness is informative)
# df_clean["age_was_missing"] = df["age"].isnull().astype(int)

print(f"Missing values after imputation: {df_clean.isnull().sum().sum()}")

## 3. Exploratory Data Analysis

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_clean[target].value_counts().plot(kind="bar", ax=axes[0], color=["salmon", "skyblue"])
axes[0].set_title("Survival Count")
axes[0].set_xticklabels(["Died", "Survived"], rotation=0)

# Survival rate by class
df_clean.groupby("pclass")[target].mean().plot(kind="bar", ax=axes[1], color="teal")
axes[1].set_title("Survival Rate by Passenger Class")
axes[1].set_ylabel("Survival Rate")
axes[1].set_xticklabels(["1st", "2nd", "3rd"], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of numerical features by survival
numerical_cols = ["age", "fare", "sibsp", "parch"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, numerical_cols):
    for survived, color, label in [(0, "salmon", "Died"), (1, "skyblue", "Survived")]:
        subset = df_clean[df_clean[target] == survived]
        ax.hist(subset[col], bins=30, alpha=0.6, color=color, label=label)
    ax.set_title(f"Distribution of {col}")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix for numerical features
numerical_df = df_clean.select_dtypes(include=[np.number])
corr = numerical_df.corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))  # mask upper triangle
sns.heatmap(corr, mask=mask, annot=True, cmap="RdBu_r", center=0, fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

## 4. Encoding Categorical Variables

In [ ]:
# Identify categorical columns
cat_cols = df_clean.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print(f"Categorical columns: {cat_cols}")
for col in cat_cols:
    print(f"  {col}: {df_clean[col].unique()}")

In [ ]:
# Binary encoding for sex (only 2 categories)
df_encoded = df_clean.copy()
df_encoded["sex"] = (df_encoded["sex"] == "male").astype(int)
df_encoded["alone"] = df_encoded["alone"].astype(int)

# One-hot encoding for embarked (nominal, few categories)
df_encoded = pd.get_dummies(df_encoded, columns=["embarked"], prefix="embarked", drop_first=True)

# Ordinal encoding for class (natural order: First > Second > Third)
class_mapping = {"First": 1, "Second": 2, "Third": 3}
df_encoded["class"] = df_encoded["class"].map(class_mapping)

df_encoded.head()

## 5. Feature Scaling

| Scaler | Formula | When to Use |
|--------|---------|------------|
| StandardScaler | (x - mean) / std | Default choice, assumes ~normal distribution |
| MinMaxScaler | (x - min) / (max - min) | When you need bounded [0,1] range |
| RobustScaler | (x - median) / IQR | When data has significant outliers |

**Critical rule**: Fit scalers on training data only, then transform both train and test.

In [ ]:
# Split BEFORE scaling to prevent data leakage
X = df_encoded.drop(columns=[target])
y = df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True).round(3)}")

In [ ]:
# Compare scalers
cols_to_scale = ["age", "fare"]

scalers = {
    "Original": None,
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
}

fig, axes = plt.subplots(len(scalers), len(cols_to_scale), figsize=(12, 10))

for i, (name, scaler) in enumerate(scalers.items()):
    if scaler is not None:
        scaled = scaler.fit_transform(X_train[cols_to_scale])
    else:
        scaled = X_train[cols_to_scale].values

    for j, col in enumerate(cols_to_scale):
        axes[i, j].hist(scaled[:, j], bins=30, color="steelblue", alpha=0.7)
        axes[i, j].set_title(f"{name} - {col}")

plt.tight_layout()
plt.show()

In [ ]:
# Apply StandardScaler (fit on train, transform both)
scaler = StandardScaler()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])  # NOT fit_transform!

print("Scaled training data sample:")
X_train.head()

## 6. Putting It All Together with sklearn Pipelines

Pipelines prevent data leakage by ensuring transformations are fit only on training data during cross-validation.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Start from the raw (un-encoded, un-scaled) data
df_raw = df[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "survived"]].copy()

X_raw = df_raw.drop(columns=["survived"])
y_raw = df_raw["survived"]

# Define transformers for different column types
numerical_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["sex", "embarked"]
passthrough_features = ["pclass"]  # already numeric ordinal

numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features),
    ("pass", "passthrough", passthrough_features),
])

# Full pipeline: preprocessing + model
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42)),
])

# Cross-validation (pipeline ensures no leakage)
scores = cross_val_score(pipeline, X_raw, y_raw, cv=5, scoring="accuracy")
print(f"Cross-validated accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

## Key Takeaways

1. **Always explore before modeling** - understand distributions, missing patterns, and correlations
2. **Handle missing values intentionally** - choose strategy based on missingness mechanism
3. **Encode categoricals appropriately** - one-hot for nominal, ordinal for ordered
4. **Scale after splitting** - fit on train only to prevent data leakage
5. **Use pipelines** - they encapsulate preprocessing + model and prevent leakage during CV